## Standards

MPEG = "Moving Picture Experts Group" - группа инженеров, собралась в 1988 году для разработки открытых стандартов формата видео.
До 2020 года они отвечали за приятие глобальных стандартов кодирования: 
- MPEG-1 1993 (для CD-дисков, mp3 для аудио)
- MPEG-2 1995 (для DVD)
- MPEG-3 (хотели делать для HD видео, но нового создавать не понадобилось => отемнили)
- MPEG-4 1999-2003 (DivX, Xvid, стандарты для 4K видео, H.264 AVC)
- MPEG-H 2013 (стандарт H.265 для iPhone)

Стандарты продолжают развиваться (например, формат H.266), но группа разделилась по-другому называется

MP3 - это аудио стандарт, разработанный еще внутри MPEG-1, а III - это степень сжатия<br>
MPEG-3 отменили не из-за созвучия с MP3

## Formats

Часто используется термин контейнер. Что это такое?<br>
Контейнер объединяет внутри себя:
1.  Видеопоток (один или несколько)
2.  Аудиодорожки (может быть несколько на разных языках)
3.  Субтитры (текстовые или графические)
4.  Метаданные (название, обложка, главы, информация о камере)

Важно понимать: расширение файла (.mp4, .mkv) говорит только о типе контейнера, но ничего не гарантирует относительно кодеков внутри. Это как коробка от обуви, в которой могут лежать инструменты.

### MP4

#### Короткое описание

mdat - непосредственно (сжатые) байты<Br>
moov - карта видео, сопоставляет время с адресом в байтах<br>нужна для навигации, и чтобы знать как распаковать (куски закодированы неравномерно)

<img src="img/mp4_1.png" width=250>

#### Более полное описание

Ниже пример mp4 контейнера

<img src="img/mp4.jpg" width=400>

Структура контейнеров (см катинку выше) описывается стандартом ISO Base Media File Format, за основу которого в сове время взяли структуру mov файлов. У разных форматов 

iods = Initial Object Descriptor - содержит описание профиля и уровня сложности файла. Это помогает плееру сразу понять: «Хватит ли мне мощности процессора, чтобы декодировать этот поток?». Здесь же указываются ID аудио и видео дорожек, которые должны стартовать одновременно.

mdhd = Media Header Box - содержит информацию, специфичную для конкретного типа медиа:
    * **Язык:** Например, "rus" или "eng" для аудиодорожки.
    * **Timescale:** Точность времени именно для этого потока.

hdlr = Handler Reference Box - объявляет тип медиа внутри `trak`. Он говорит системе: «Для обработки этого блока данных вызывай видео-декодер» (код `vide`) или «вызывай аудио-декодер» (код `soun`)

vmhd / smhd = Video/Sound Media Header - содержит режим наложения (graphics mode) и цвета для композитинга или баланс звука (моно/стерео) и другие базовые аудио-параметры

dinf и dref = Data Information & Reference - позволяют MP4-файлу быть «пустым». Блок `dref` может содержать URL, указывающий на другой файл. То есть ваш `.mp4` может быть просто маленьким файлом-ссылкой, который говорит плееру: «Карта данных здесь, а сами гигабайты видео возьми в файле по адресу `http://...`»

stsd = Sample Description - «переводчик» для данных. Если `mdat` — это просто сжатые байты, то `stsd` объясняет, как их понимать.
На скриншоте видно внутри блок **avc1**. Это означает, что видео сжато кодеком **H.264 (AVC). Внутри него есть **avcC** — это конфигурация кодека (параметры профиля, уровня и т.д.), без которой декодер не сможет даже начать обработку первого кадра

pasp = Pixel Aspect Ratio - он говорит плееру, являются ли пиксели квадратными. Например, старые форматы могли хранить видео 720x576, но растягивать его до 1024x576 при показе. `pasp` отвечает за это растяжение.

ctts = Composition Time to Sample - Видеокадры в современных кодеках (H.264/H.265) часто хранятся **не в том порядке**, в котором показываются (из-за B-кадров, которые ссылаются на будущие кадры). `ctts` — это таблица смещения, которая говорит плееру: «Декодируй этот кадр сейчас, но покажи его только через 2 кадра»

sdtp = Sample Dependency Type - описывает зависимости кадров. Какие кадры можно выбросить без вреда для остальных (например, при перемотке), а какие являются критически важными (I-кадры)

edts и elst = Edit List - позволяют делать нелинейный монтаж прямо внутри контейнера. Например, можно сказать плееру: «Проиграй первые 5 секунд, потом прыгни сразу на 20-ю секунду, а звук при этом не прерывай».

Есть утилиты `mediainfo` или `ffprobe` вытащить эти данные из любого твоего видеофайла в текстовом виде?

Карта видео - это набор таблиц,
- stts (Time-to-Sample) длительность каждого кадра<br>
* *Пример:* «Кадр №1 длится 0.04 сек, кадр №2 длится 0.04 сек...» (так вычисляется FPS).
- stss (Sync Sample Table) — Ключевые кадры (I-frames), из-за сжатия начать можно только с них
- stsc (Sample-to-Chunk) Данные в файле группируются в «чанки» (куски). Эта таблица говорит, сколько кадров упаковано в каждый чанк.
- stsz (Sample Size) Видео сжимается неравномерно: один кадр (сложный) может весить 100 КБ, а другой (простой) — 2 КБ.
- stco (Chunk Offset) — Хранит точный адрес каждого чанка в байтах от начала файла. 

### Алгоритм H.264

H.264 (AVC — Advanced Video Coding) уменьшает объем данных в **100–1000 раз**

Кодек работает на двух уровнях сжатия:
1.  **Внутрикадровое (Intra):** Сжимает каждый кадр как отдельную картинку (аналог JPEG).
2.  **Межкадровое (Inter):** Самое важное. Кодек анализирует изменения между кадрами. Если вы стоите на фоне стены и машете рукой, кодек запишет стену один раз, а для остальных кадров будет записывать только движение руки.

Алгоритм H.264 делит видео на цепочки кадров трех типов:

* **I-кадры (Intra):** "Опорные" кадры. Это полные изображения. Они самые тяжелые. С них начинается любой фрагмент видео.
* **P-кадры (Predicted):** "Предсказанные" кадры. Они хранят только изменения относительно предыдущего I или P кадра.
* **B-кадры (Bi-predictive):** Самые эффективные. Они смотрят и назад (в прошлое), и вперед (в будущее), чтобы вычислить среднее значение пикселей.

Когда вы нажимаете "Кодировать" в FFmpeg, происходит следующее:

1. сетка из 16x16 блоков
2. Кодек ищет похожий блок в предыдущем кадре. Если он нашел, что блок "дерево" просто сместился на 5 пикселей вправо, он не сохраняет пиксели дерева снова. Он записывает **вектор движения**: «Блок №5, сместись на [5, 0]»
3. Кодек вычитает "предсказанное" изображение из реального. Получается "картинка-призрак", где видны только ошибки предсказания. Именно этот крошечный остаток и нужно сжать
4. Математическая магия. Данные из пространства (пиксели) переводятся в пространство частот. 
* Человеческий глаз плохо видит мелкие изменения яркости в деталях. 
* DCT позволяет отбросить высокие частоты (шум, микро-детали), которые мы всё равно не заметим.
5. Это стадия, где происходит реальная потеря качества. Мы округляем значения после DCT. 
* Если округляем сильно — файл крошечный, но появляются "квадраты" (артефакты).
* Если слабо — качество как в оригинале, но файл большой.
6. Финальный этап сжатия без потерь (как в ZIP-архиве). Часто встречающиеся комбинации битов заменяются на короткие коды, а редкие — на длинные

---

### Пример команды FFmpeg для идеального сжатия в H.264:

```bash
ffmpeg -i input.mov -c:v libx264 -crf 23 -preset medium -c:a aac -b:a 128k output.mp4

## FFMpeg

**FFmpeg** — это ведущий мультимедийный фреймворк с открытым исходным кодом, созданный в 2000 году под Linux как универсальное решение для декодирования и кодирования любых форматов мультимедиа

Автор - программист, также создавший эмулятор QEMU () и быстрый компилятор TinyCC (). 
Префикс "FF" означает "Fast Forward", а MPEG - это стандарт, для которого он вначале был предназначен



Цель любого кодека - обеспечить:
- универсальность формата для портабильности
- хорошей сжатие для эффективной работы

---

### 2. Развитие функциональности во времени

Проект развивался вместе с индустрией цифрового видео:

* **2000-е:** Эпоха DVD и раннего интернета. Поддержка MPEG-2, DivX, Xvid. Фокус на базовой конвертации.
* **2010-е:** Расцвет стриминга и HD. Добавлена мощная поддержка H.264 (libx264), аппаратного ускорения (NVENC, QuickSync) и протоколов вещания (RTMP, HLS).
* **2020-е:** Переход на 4K/8K и нейросети. Внедрение поддержки кодеков AV1, глубокая интеграция с библиотеками машинного обучения для улучшения видео (фильтры на базе ИИ) и поддержка HDR.

---

### 3. Основная функциональность

FFmpeg состоит из нескольких библиотек (`libavcodec`, `libavformat` и др.), которые позволяют выполнять следующие задачи:

1.  **Транскодирование:** Преобразование из одного формата в другой (например, AVI в MP4).
2.  **Демьюксинг/Мьюксинг:** Разделение файла на аудио и видео дорожки и их сборка обратно.
3.  **Фильтрация:** Масштабирование, обрезка, коррекция цвета, наложение водяных знаков.
4.  **Стриминг:** Передача видеопотока в реальном времени на серверы (YouTube, Twitch).
5.  **Захват:** Запись видео с экрана или вебкамеры.



---

### 4. Мини-туториал: Ключевые команды

Ниже приведен список самых полезных команд для повседневной работы.

#### Информация о файле
```bash
ffprobe input.mp4